# Gemma 4 E2B 1000-position campaign: Worker 2 of 2 (positions [500:1000])

Local Gemma 4 E2B, thinking ON, on the SAME 1000 positions the DeepSeek direct-mode run scored (48.6% strict). Two GPU workers cover 500 positions each (Kaggle's 2-concurrent-GPU limit). Methodology identical to the validated check10 kernel: `--local-thinking`, `--force-answer-prompt`, `--max_new_tokens 32768`.

Backups: HF upload every 25 positions; the recovery cell pulls this worker's checkpoint back before `--resume`, so a died session relaunches and continues. Live: streams to monitor/gemma/workers/w{n}.* -- the aggregator kernel combines both workers into the gemma dashboard page.

## 1. Secrets (hardcoded env vars)

In [ ]:
import os
print("secrets are injected at build time; this cell is a placeholder")

## 2. Get the repo (mate-e2b-kaggle branch)

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "chess-slm-benchmark"
if REPO.exists():
    shutil.rmtree(REPO)

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

token = find_token()
url = "https://github.com/Vedang-P/chess-slm-benchmark.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
# --local-thinking, --live-namespace, scored-state live publishing and the
# mid-stream thinking split live on the mate-e2b-kaggle branch; clone the
# branch explicitly so the kernel always runs the intended code.
res = subprocess.run(["git", "clone", "--quiet", "--branch", "mate-e2b-kaggle",
                      url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("clone failed (token not attached?): " + res.stderr[-300:])
os.chdir(REPO)
print("cwd:", Path.cwd())

## 3. Dependencies (torch pins for P100+T4)

In [ ]:
import subprocess, sys
# Kaggle's free tier hands out a P100 (sm_60) OR a T4 (sm_75). Recent torch
# wheels dropped sm_60, so pin the last CUDA-12.1 build with both archs
# BEFORE anything else installs torch; torchvision/torchaudio must match the
# pinned torch (Kaggle ships torchvision built for the latest torch -- the
# ABI mismatch crashes transformers' AutoProcessor import). bitsandbytes is
# pinned to the matching multi-CUDA build, and the requirements install
# afterwards must NOT clobber these pins (no -U: it still upgrades
# transformers to >=5.13 on its own).
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "torch==2.5.1", "--index-url",
                "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "torchvision==0.20.1", "torchaudio==2.5.1", "--index-url",
                "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "bitsandbytes==0.44.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-r", "requirements.txt"], check=True)
import torch, transformers
if int(transformers.__version__.split(".")[0]) < 5:
    raise RuntimeError(f"transformers {transformers.__version__} too old "
                       "for Gemma 4 (needs >= 5.13)")
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          "| cap", torch.cuda.get_device_capability(0),
          "| vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 4. Engine/dataset gate

In [ ]:
status = subprocess.run([sys.executable, "scripts/test_engine.py", "--quick"],
                        capture_output=True, text=True)
if status.returncode != 0:
    print(status.stdout[-2000:]); print(status.stderr[-2000:])
    raise RuntimeError("test_engine failed")
print("ALL TESTS PASSED")

## 5. Demo: 2 positions through the REAL pipeline

Verifies model load, the thinking channel split, extraction, live push and the HF upload all work on the GPU Kaggle assigned (T4 or P100 -- the pinned torch 2.5.1 build supports both). The demo shares the run id and output dir with the full run, so the full run's `--resume` skips these 2 positions and the HF archive ends up with one complete cell.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

# 2-position demo through the REAL pipeline: model load, thinking channel,
# extraction, live push, HF end-upload -- everything the full run does, on
# whichever GPU Kaggle assigned (T4 or P100; both are supported by the
# pinned torch build). Shares the run id + output dir with the full run, so
# the archive ends up with ONE cell and the full run resumes past these 2.
os.environ["BENCH_RUN_ID"] = 'gemma1000-w2'
out = Path('results/gemma-1000-w2')
out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, "scripts/run_mate_eval.py",
       "--model", "gemma4-e2b",
       "--offset", "500",
       "--n", "2",
       "--max_new_tokens", "32768",
       "--force-answer-prompt",
       "--local-thinking",
       "--worker-id", 'w2',
       "--live-namespace", "gemma",
       "--output_dir", 'results/gemma-1000-w2',
       "--live-push",
       "--resume",
       "--verbose"]
print("demo:", " ".join(cmd))
t0 = time.time()
res = subprocess.run(cmd, stderr=subprocess.STDOUT)
print(f"demo exit rc={res.returncode} after {(time.time()-t0)/60:.1f}min")
if res.returncode != 0:
    raise RuntimeError(f"demo failed with rc={res.returncode} -- fix before the full run")

## 6. Inspect the demo before the full run

Raises if anything is broken: no samples written, 0 parsed answers, or missing reasoning text (thinking split failed). On a relaunch after a died session this cell validates the recovered samples instead.

In [ ]:
import json, glob
from pathlib import Path

files = glob.glob(str(Path('results/gemma-1000-w2') / "*.samples.jsonl"))
if not files:
    raise RuntimeError(f"no samples file under 'results/gemma-1000-w2' -- the demo did not write results")
rows = [json.loads(l) for l in open(files[0]) if l.strip()]
print(f"demo samples on disk: {len(rows)}")
for s in rows:
    tu = s.get("token_usage") or {}
    print(f"{s['position_id']}: {s['status']:9s} move={s.get('move')} "
          f"correct={s.get('compliance')} reasoning_chars={s.get('reasoning_chars')} "
          f"reason_tokens={tu.get('reasoning_tokens')}")
parsed = [r for r in rows if r["status"] in ("correct", "wrong")]
unclean = [r for r in rows if not (r.get("reasoning") or "").strip()]
if not parsed:
    raise RuntimeError("0 parsed positions -- extraction/thinking split is broken on this GPU")
if unclean:
    raise RuntimeError(f"{len(unclean)} sample(s) have no reasoning text -- the thinking "
                       "channel split is broken; check the samples above")
print(f"\nDEMO OK: {len(parsed)}/{len(rows)} parsed, thinking split verified -- safe to run the full slice")

## 7. Run positions [500:1000]

HF backup every 25 positions; live state to monitor/gemma/workers/w2.* (the aggregator kernel combines both workers into the dashboard).

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

os.environ["BENCH_RUN_ID"] = 'gemma1000-w2'
out = Path('results/gemma-1000-w2')
out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, "scripts/run_mate_eval.py",
       "--model", "gemma4-e2b",
       "--offset", "500",
       "--n", "500",
       "--max_new_tokens", "32768",
       "--force-answer-prompt",
       "--local-thinking",
       "--worker-id", 'w2',
       "--live-namespace", "gemma",
       "--output_dir", 'results/gemma-1000-w2',
       "--live-push",
       "--hf-upload-every", "25",
       "--resume",
       "--verbose"]
print("running:", " ".join(cmd))
t0 = time.time()
# unbuffered + merged stderr so a crash's real traceback lands in THIS
# cell's own output instead of vanishing into the subprocess's stderr pipe.
res = subprocess.run(cmd, stderr=subprocess.STDOUT)
elapsed_h = (time.time() - t0) / 3600
print(f"run exited rc={res.returncode} after {elapsed_h:.2f}h")
if res.returncode != 0:
    raise RuntimeError(
        f"run_mate_eval.py exited rc={res.returncode} after only "
        f"{elapsed_h:.2f}h -- a real failure, not a completed run. See the "
        "subprocess output printed above for the traceback."
    )

## 8. This worker's summary

In [ ]:
import json, glob, collections
from pathlib import Path

rows = []
for f in glob.glob(str(Path('results/gemma-1000-w2') / "*.samples.jsonl")):
    for line in open(f):
        if line.strip():
            rows.append(json.loads(line))
n = len(rows)
by_status = collections.Counter(r["status"] for r in rows)
correct = sum(bool(r["compliance"]) for r in rows if r["status"] != "api_error")
scored = n - by_status.get("api_error", 0)
print(f"this worker: {n} / 500 positions attempted")
print("status breakdown:", dict(by_status))
if scored:
    print(f"accuracy (of {scored} scored): {correct}/{scored} = {correct/scored:.3f}")
print()
print("DeepSeek reference (direct mode, 1000 positions): 48.6% strict")

## Notes
- Scope: ONLY positions [500:1000) of mate-selection-test.json. The other worker owns the rest.
- If this session dies/times out, push the same notebook again (or Restart & Run All): the demo/inspect cells are no-ops once the slice is done, step 5 recovers progress from HF, --resume skips it.
- GPU: Kaggle assigns T4 or P100 -- both supported; T4 is ~2x faster. ~78s/position measured on P100 (check10), so 500 positions ~ 11h, within the 12h session limit.
- Results: HF runs/gemma1000-w2/; live: chess-bench-live.pages.dev/gemma.html